# NBA 球员数据监督学习建模 —— Baseline 基线模型

本 notebook 基于 `../data/processed/master_data.csv` 完成两类监督学习任务：

1. **回归任务**：用当季技术统计预测球员 **下一赛季的 PER**（Baseline：`LinearRegression` / `DecisionTreeRegressor`）；
2. **分类任务**：根据当季高阶统计判断球员是否达到 **全明星实力门槛**（Baseline：`LogisticRegression`）。

为防止时间序列上的未来数据泄露，训练集与测试集严格按赛季年份切分：
- 训练集：`Year <= 2010`
- 测试集：`Year > 2010`

## 1. 数据准备与读取

### 1.1 导入依赖库

In [2]:
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

print(f"pandas        : {pd.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")

pandas        : 2.3.3
scikit-learn  : 1.9.0


### 1.2 读取主数据集

In [3]:
DATA_PATH = "../data/processed/master_data.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"主数据集规模: {df_raw.shape[0]} 行 x {df_raw.shape[1]} 列")
print(f"赛季覆盖范围: {df_raw['Year'].min()} - {df_raw['Year'].max()}")
print(f"球员数量    : {df_raw['Player'].nunique()}")

主数据集规模: 20313 行 x 78 列
赛季覆盖范围: 1950 - 2017
球员数量    : 3921


### 1.3 剔除无关 / 易泄露字段

回归任务的标签 `PER_next` 需要先按球员分组做跨赛季移位，因此必须在删除 `Player` 之前构造；随后再统一剔除身份标识类、球队环境类与出生信息类字段。

In [4]:
# ---- 先构造回归标签：PER_next = 该球员下一赛季的 PER ----
df = df_raw.sort_values(["Player", "Year"]).reset_index(drop=True)
df["PER_next"] = df.groupby("Player")["PER"].shift(-1)

# ---- 剔除与建模无关或可能引起数据泄露的字段 ----
# Player/Tm：身份与环境标识；college/birth_*：背景与出生信息；
# year_start/year_end/position_career：生涯汇总信息，不应作为单赛季特征。
drop_cols = [
    "Player",
    "Tm",
    "college",
    "birth_year",
    "birth_city",
    "birth_state",
    "birth_date",
    "year_start",
    "year_end",
    "position_career",
]

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

print(f"剔除后数据规模: {df_model.shape[0]} 行 x {df_model.shape[1]} 列")
print(f"剔除字段: {drop_cols}")

剔除后数据规模: 20313 行 x 69 列
剔除字段: ['Player', 'Tm', 'college', 'birth_year', 'birth_city', 'birth_state', 'birth_date', 'year_start', 'year_end', 'position_career']


### 1.4 Pos 位置字段：One-Hot Encoding

位置 `Pos` 是类别型字段，这里用 `pd.get_dummies` 生成哑变量；后续回归与分类任务使用**同一套统计特征 + Pos 哑变量**。

In [5]:
# 两任务共用的当季统计特征（与任务要求保持一致）
feature_cols = [
    "Age",
    "G",
    "MP",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "PTS_per36",
    "AST%",
    "TRB%",
    "WS",
]

# Pos -> One-Hot Encoding
pos_dummies = pd.get_dummies(df_model["Pos"], prefix="Pos").astype(int)
pos_cols = pos_dummies.columns.tolist()

df_model = pd.concat(
    [df_model.drop(columns=["Pos"]), pos_dummies],
    axis=1,
)

print(f"Pos 独热编码后新增 {len(pos_cols)} 个哑变量列")
print(f"哑变量列示例: {pos_cols[:5]} ...")
print(f"统计特征: {feature_cols}")

Pos 独热编码后新增 23 个哑变量列
哑变量列示例: ['Pos_C', 'Pos_C-F', 'Pos_C-PF', 'Pos_C-SF', 'Pos_F'] ...
统计特征: ['Age', 'G', 'MP', 'TS%', '3PAr', 'FTr', 'USG%', 'PTS_per36', 'AST%', 'TRB%', 'WS']


## 2. 构造两大预测任务的标签 (y) 与特征矩阵 (X)

### 2.1 公共特征矩阵构建函数

In [6]:
def build_X(data: pd.DataFrame) -> pd.DataFrame:
    """返回两任务共用的当季统计特征矩阵（统计特征 + Pos 哑变量）。"""
    return data[feature_cols + pos_cols].copy()

### 2.2 任务一：回归 —— 预测球员【下一个赛季】的 PER

- 标签：`PER_next`（`shift(-1)` 得到下一赛季 PER）；
- 删除 `PER_next` 为 NaN 的记录（最后一个赛季无法匹配下一年数据）；
- 特征：当季的 `Age, G, MP, TS%, 3PAr, FTr, USG%, PTS_per36, AST%, TRB%, WS` 及 `Pos` 哑变量。

In [7]:
reg_df = df_model[df_model["PER_next"].notna()].copy()

X_reg = build_X(reg_df)
y_reg = reg_df["PER_next"]
year_reg = reg_df["Year"]

print(f"回归任务样本数: {X_reg.shape[0]}")
print(f"特征矩阵 X_reg: {X_reg.shape[0]} 行 x {X_reg.shape[1]} 列")
print(f"样本覆盖赛季  : {year_reg.min()} - {year_reg.max()}")

回归任务样本数: 16260
特征矩阵 X_reg: 16260 行 x 34 列
样本覆盖赛季  : 1950 - 2016


### 2.3 任务二：分类 —— 判断球员【当赛季】是否达到全明星实力门槛

- 标签：`Is_AllStar_Caliber`，当 `PER >= 20.0` 且 `WS >= 6.0` 时为 1，否则为 0；
- 特征：与回归任务保持一致的当季统计特征（`X_cls`）。

> 标签依赖 `PER` 与 `WS`，两字段缺失时无法可靠判定，先剔除对应记录。
>
> **注意**：按题目要求 `X_cls` 与 `X_reg` 完全一致（包含 `WS`），而标签阈值本身也依赖 `WS`，严格意义上存在规则重叠/泄露；此处保留该设定以完成 Baseline，后续建模可改用真实全明星名单并剔除 `PER` / `WS`。

In [8]:
cls_df = df_model.dropna(subset=["PER", "WS"]).copy()

cls_df["Is_AllStar_Caliber"] = (
    (cls_df["PER"] >= 20.0) & (cls_df["WS"] >= 6.0)
).astype(int)

X_cls = build_X(cls_df)
y_cls = cls_df["Is_AllStar_Caliber"]
year_cls = cls_df["Year"]

print(f"分类任务样本数: {X_cls.shape[0]}")
print(f"特征矩阵 X_cls: {X_cls.shape[0]} 行 x {X_cls.shape[1]} 列")
print(f"正类（全明星实力）占比: {y_cls.mean():.4%}")
print(f"样本覆盖赛季  : {year_cls.min()} - {year_cls.max()}")

分类任务样本数: 19921
特征矩阵 X_cls: 19921 行 x 34 列
正类（全明星实力）占比: 5.3009%
样本覆盖赛季  : 1952 - 2017


## 3. 数据集划分（时间维度切分）

为防止模型看到“未来赛季”的信息，这里不进行随机切分，而是严格按照年份：
- **训练集 Train**：`Year <= 2010`；
- **测试集 Test**：`Year > 2010`。

In [9]:
def time_split(X, y, years, train_max=2010):
    """按赛季年份切分：训练集取 years <= train_max，测试集取 years > train_max。"""
    train_mask = years <= train_max
    test_mask = ~train_mask
    return (
        X.loc[train_mask],
        X.loc[test_mask],
        y.loc[train_mask],
        y.loc[test_mask],
    )


# ---- 回归任务划分 ----
X_train_reg, X_test_reg, y_train_reg, y_test_reg = time_split(X_reg, y_reg, year_reg)
train_years_reg = year_reg[year_reg <= 2010]
test_years_reg = year_reg[year_reg > 2010]

# ---- 分类任务划分 ----
X_train_cls, X_test_cls, y_train_cls, y_test_cls = time_split(X_cls, y_cls, year_cls)
train_years_cls = year_cls[year_cls <= 2010]
test_years_cls = year_cls[year_cls > 2010]

print("【回归】训练集:", X_train_reg.shape, "| 赛季:", train_years_reg.min(), "-", train_years_reg.max())
print("【回归】测试集:", X_test_reg.shape, "| 赛季:", test_years_reg.min(), "-", test_years_reg.max())
print()
print("【分类】训练集:", X_train_cls.shape, "| 赛季:", train_years_cls.min(), "-", train_years_cls.max())
print("【分类】测试集:", X_test_cls.shape, "| 赛季:", test_years_cls.min(), "-", test_years_cls.max())
print()
print(
    f"分类正类占比 -> 训练集: {y_train_cls.mean():.4%} | 测试集: {y_test_cls.mean():.4%}"
)

【回归】训练集: (13883, 34) | 赛季: 1950 - 2010
【回归】测试集: (2377, 34) | 赛季: 2011 - 2016

【分类】训练集: (16587, 34) | 赛季: 1952 - 2010
【分类】测试集: (3334, 34) | 赛季: 2011 - 2017

分类正类占比 -> 训练集: 5.1788% | 测试集: 5.9088%


## 4. 模型构建与评估 (Baseline)

### 4.1 回归模型：LinearRegression & DecisionTreeRegressor

使用 `ColumnTransformer` 构建预处理流程：
- 数值统计特征：`SimpleImputer(strategy="median")` 中位数填补 → `StandardScaler` 标准化；
- `Pos` 哑变量：原样透传（已经为 0/1 编码）。

评估指标：测试集上的 **R²** 与 **RMSE**。

In [10]:
def make_reg_preprocessor():
    """构建回归模型共用的预处理流程（每次调用返回新实例）。"""
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, feature_cols),
            ("pos", "passthrough", pos_cols),
        ],
        remainder="passthrough",
    )


reg_models = {
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=5, random_state=42),
}

for name, estimator in reg_models.items():
    model = Pipeline([
        ("preprocessor", make_reg_preprocessor()),
        ("estimator", estimator),
    ])

    model.fit(X_train_reg, y_train_reg)
    y_pred_reg = model.predict(X_test_reg)

    r2 = r2_score(y_test_reg, y_pred_reg)
    rmse = float(np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)))

    print(f"[回归] {name}")
    print(f"  R2   = {r2:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print()

[回归] LinearRegression
  R2   = 0.3441
  RMSE = 4.7785

[回归] DecisionTreeRegressor
  R2   = 0.2839
  RMSE = 4.9930



### 4.2 分类模型：LogisticRegression

使用 `Pipeline` 完成：中位数填补 → 标准化 → `LogisticRegression(random_state=42)`。

评估指标：测试集上的 **Accuracy、Precision、Recall、F1-Score**，并输出 `classification_report`。

In [11]:
cls_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
])

cls_pipeline.fit(X_train_cls, y_train_cls)
y_pred_cls = cls_pipeline.predict(X_test_cls)

accuracy = accuracy_score(y_test_cls, y_pred_cls)
precision = precision_score(y_test_cls, y_pred_cls)
recall = recall_score(y_test_cls, y_pred_cls)
f1 = f1_score(y_test_cls, y_pred_cls)

print("[分类] LogisticRegression")
print(f"  Accuracy  = {accuracy:.4f}")
print(f"  Precision = {precision:.4f}")
print(f"  Recall    = {recall:.4f}")
print(f"  F1-Score  = {f1:.4f}")
print()
print("classification_report:")
print(classification_report(y_test_cls, y_pred_cls, digits=4))

[分类] LogisticRegression
  Accuracy  = 0.9745
  Precision = 0.9308
  Recall    = 0.6142
  F1-Score  = 0.7401

classification_report:
              precision    recall  f1-score   support

           0     0.9763    0.9971    0.9866      3137
           1     0.9308    0.6142    0.7401       197

    accuracy                         0.9745      3334
   macro avg     0.9535    0.8057    0.8633      3334
weighted avg     0.9736    0.9745    0.9720      3334



## 5. Baseline 小结

- 回归：`LinearRegression` 与 `DecisionTreeRegressor(max_depth=5, random_state=42)` 已按年份切分完成训练与测试评估（R²、RMSE）；
- 分类：`LogisticRegression` 已基于当季特征完成全明星实力二分类评估（Accuracy / Precision / Recall / F1）。

后续可在同一时间切分与预处理框架下继续升级为 Random Forest、XGBoost、LightGBM 等模型，并结合交叉验证与 SHAP 进行特征解释。